# Spark Structured Streaming — exemplo prático

Neste notebook vamos processar dados **em tempo real** com Spark Structured Streaming.

**Ideias principais**

- Um stream é um DataFrame que **cresce para sempre**. Você escreve as transformações como se fosse um batch e o Spark executa de forma incremental.
- O Spark processa em **micro-batches**: a cada intervalo (o *trigger*) ele pega os dados novos, atualiza o resultado e entrega ao *sink* (destino).
- **Output mode** define o que é escrito a cada batch: `append` (só linhas novas), `update` (só linhas alteradas) ou `complete` (o resultado inteiro).
- **Event time** é o instante em que o evento aconteceu. Agregações por janela de tempo usam essa coluna, e o **watermark** diz até quando eventos atrasados ainda são aceitos.

Para não depender de Kafka ou de arquivos compartilhados, os exemplos usam a fonte `rate`, que gera linhas com `timestamp` e `value` em uma taxa configurável.

## Setup

A sessão aponta para o cluster `spark-master`. O PySpark já vem instalado na imagem do Jupyter, então basta importá-lo.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("streaming-exemplo")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/30 14:12:19 WARN Utils: Your hostname, <hostname>, resolves to a loopback address: 127.0.1.1; using <host-ip> instead (on interface wlo1)
26/10/30 14:12:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 14:12:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/30 14:12:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/30 14:12:20 WARN StandaloneAppClient$ClientEndpoint: Failed to connect to master spark-master:7077
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkT

KeyboardInterrupt: 

26/10/30 14:13:20 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: All masters are unresponsive! Giving up.
26/10/30 14:13:21 WARN StandaloneAppClient$ClientEndpoint: Drop UnregisterApplication(null) because has not yet connected to master
26/10/30 14:13:21 WARN MetricsSystem: Stopping a MetricsSystem that is not running


In [ ]:
def parar_streams():
    """Para todas as queries de streaming ativas nesta sessão."""
    for q in spark.streams.active:
        q.stop()
    print("Queries ativas:", len(spark.streams.active))

## Exemplo 1 — O básico: stream para o console

O `readStream` cria o DataFrame de entrada. Como a fonte `rate` gera 5 linhas por segundo, o `show` do stream aparece no console a cada 2 segundos.

`awaitTermination(12)` deixa a query rodar por 12 segundos e depois devolve o controle para o notebook.

In [ ]:
stream_basico = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

query_console = (
    stream_basico.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .trigger(processingTime="2 seconds")
    .start()
)

query_console.awaitTermination(12)
query_console.stop()

## Exemplo 2 — Pedidos em tempo real com janelas e enriquecimento

Cenário: uma loja recebe pedidos continuamente. Vamos:

1. Gerar pedidos simulados (`order_id`, `order_ts`, `product_id`, `quantity`).
2. Enriquecer cada pedido com a categoria e o preço do produto. O catálogo `products.csv` é uma tabela **estática** (batch), e fazer join de stream com estática é um padrão muito comum.
3. Calcular a receita por categoria em **janelas de 10 segundos que deslizam a cada 5 segundos** (sliding window).
4. Guardar o resultado em uma tabela em memória (`memory` sink) para consultar com SQL enquanto o stream roda.

In [ ]:
# Tabela estática com o catálogo de produtos (montada em /data pelo docker-compose)
schema_produtos = T.StructType([
    T.StructField("product_id", T.IntegerType()),
    T.StructField("product_name", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("unit_price", T.DoubleType()),
])

produtos = spark.read.csv("/data/products.csv", header=True, schema=schema_produtos)
n_produtos = produtos.count()

produtos.show(5, truncate=False)
print("Produtos no catálogo:", n_produtos)

In [ ]:
# Stream de pedidos simulados
pedidos = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .load()
    .select(
        F.col("value").alias("order_id"),
        F.col("timestamp").alias("order_ts"),
        ((F.col("value") % n_produtos) + 1).cast("int").alias("product_id"),
        (F.col("value") % 4 + 1).alias("quantity"),
    )
)

# Stream-static join: cada pedido recebe categoria e preço do catálogo
pedidos_enriquecidos = (
    pedidos
    .join(produtos, "product_id")
    .withColumn("revenue", F.col("quantity") * F.col("unit_price"))
)

pedidos_enriquecidos.printSchema()

In [ ]:
# Receita por categoria em janelas de 10s, deslizando a cada 5s.
# O watermark de 30s diz que eventos com mais de 30s de atraso podem ser descartados.
receita_por_categoria = (
    pedidos_enriquecidos
    .withWatermark("order_ts", "30 seconds")
    .groupBy(F.window("order_ts", "10 seconds", "5 seconds"), "category")
    .agg(
        F.count("*").alias("pedidos"),
        F.round(F.sum("revenue"), 2).alias("receita"),
    )
)

query_receita = (
    receita_por_categoria.writeStream
    .queryName("receita_por_categoria")   # nome da tabela em memória
    .format("memory")
    .outputMode("complete")               # agregações com janela usam complete ou update
    .trigger(processingTime="5 seconds")
    .start()
)

print("Query iniciada:", query_receita.id)

Deixe o stream rodar por alguns segundos. Depois, consulte a tabela em memória com SQL. Ela é atualizada a cada micro-batch.

In [ ]:
query_receita.awaitTermination(25)

spark.sql("""
    SELECT window.start AS inicio, category, pedidos, receita
    FROM receita_por_categoria
    ORDER BY inicio DESC, receita DESC
    LIMIT 15
""").show(truncate=False)

In [ ]:
# Métricas da última micro-batch
progresso = query_receita.lastProgress
if progresso:
    print("Batch:", progresso["batchId"])
    print("Linhas de entrada:", progresso["numInputRows"])
    print("Linhas/segundo:", round(progresso["processedRowsPerSecond"], 1))
else:
    print("Nenhum batch concluído ainda.")

In [ ]:
query_receita.awaitTermination(10)
print("Ativa?", query_receita.isActive)

Quando terminar, pare as queries para liberar os recursos do cluster.

In [ ]:
parar_streams()

## Resumo

| Conceito | O que faz | Exemplo neste notebook |
|---|---|---|
| `readStream` | Cria um DataFrame de entrada que cresce com o tempo | `format("rate")`, `pedidos` |
| Transformações | Funcionam como em batch (select, filter, join, groupBy) | `join` com `produtos` |
| Trigger | Frequência dos micro-batches | `processingTime="5 seconds"` |
| Output mode | O que é escrito a cada batch | `complete` |
| Sink | Destino dos resultados | `console`, `memory` |
| Janela + watermark | Agregação por tempo de evento com tolerância a atrasos | `window(...)` + `withWatermark` |

**Em produção**, a fonte costuma ser Kafka, e o sink, um lakehouse (Delta, Iceberg) ou Kafka de novo. Também é obrigatório definir `checkpointLocation` nas queries, para o Spark conseguir retomar após falhas. Exemplo de leitura de Kafka (não executado aqui, pois o projeto não tem um tópico configurado):

```python
df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "pedidos")
    .load()
)
```

## Exercícios

1. Troque a janela deslizante por uma janela *tumbling* de 1 minuto (`window("order_ts", "1 minute")`) e rode de novo.
2. Use `outputMode("update")` no lugar de `complete`. O que muda no resultado?
3. Adicione um filtro `quantity >= 3` antes do `groupBy` e compare a receita por categoria.